<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/EarningsLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install scipy==1.16.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 38.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [24]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta , date
from math import sqrt
print("Libraries installed successfully!")

Libraries installed successfully!


In [22]:
# =============================================================
# GOOGLE DRIVE SETUP
# =============================================================

def mount_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Google Drive mounted successfully.")
    except Exception as e:
        print(f"Drive mount failed: {e}")


# Folder on your Drive where all CSVs are stored
DRIVE_FOLDER       = "/content/drive/MyDrive/EarningsLab"
PREDICTIONS_CSV    = f"{DRIVE_FOLDER}/earnings_predictions.csv"
RESULTS_CSV        = f"{DRIVE_FOLDER}/earnings_results.csv"
PERFORMANCE_CSV    = f"{DRIVE_FOLDER}/earnings_performance.csv"


def ensure_folder():
    os.makedirs(DRIVE_FOLDER, exist_ok=True)

In [34]:

# =============================================================
# WIN / LOSS LOGIC
# =============================================================

def determine_win_loss(structure, actual_move_pct, implied_move_pct,
                       directional_bias, actual_direction):
    """
    Determine Win/Loss/Partial for each suggested structure
    based on actual 2-day move after earnings.

    Parameters
    ----------
    structure         : str  — suggested options structure
    actual_move_pct   : float — absolute % move 2 days after earnings
    implied_move_pct  : float — what options priced in before earnings
    directional_bias  : str  — Bullish or Bearish (from up/down count)
    actual_direction  : str  — Up or Down (what actually happened)

    Returns
    -------
    result : str  — Win / Loss / Partial / N/A
    notes  : str  — plain English explanation
    """

    # ── Iron Condor / Sell Volatility
    # Win if actual move < implied move (stock stayed inside the range)
    if structure in ["Short Iron Condor"]:
        if actual_move_pct < implied_move_pct * 0.80:
            return "Win", (
                f"Stock moved {actual_move_pct:.1f}% vs implied {implied_move_pct:.1f}% "
                f"— stayed well inside range. Premium collected."
            )
        elif actual_move_pct < implied_move_pct:
            return "Partial", (
                f"Stock moved {actual_move_pct:.1f}% vs implied {implied_move_pct:.1f}% "
                f"— inside range but close. Partial profit likely."
            )
        else:
            return "Loss", (
                f"Stock moved {actual_move_pct:.1f}% vs implied {implied_move_pct:.1f}% "
                f"— broke outside range. Iron condor likely breached."
            )

    # ── Long Straddle / Strangle / Buy Volatility
    # Win if actual move > implied move (stock moved more than priced in)
    elif structure in ["Long Straddle", "Long Straddle / Strangle"]:
        if actual_move_pct > implied_move_pct * 1.20:
            return "Win", (
                f"Stock moved {actual_move_pct:.1f}% vs implied {implied_move_pct:.1f}% "
                f"— exceeded implied move by 20%+. Straddle profitable."
            )
        elif actual_move_pct > implied_move_pct:
            return "Partial", (
                f"Stock moved {actual_move_pct:.1f}% vs implied {implied_move_pct:.1f}% "
                f"— beat implied move but not enough to cover full premium. Marginal."
            )
        else:
            return "Loss", (
                f"Stock moved {actual_move_pct:.1f}% vs implied {implied_move_pct:.1f}% "
                f"— did not exceed implied move. Straddle expired worthless or at loss."
            )

    # ── Directional / Calendar Spread
    # Win if direction matched the directional bias
    elif structure in ["Directional or Calendar Spread"]:
        if actual_direction == actual_direction and directional_bias.lower() in actual_direction.lower():
            return "Win", (
                f"Stock moved {actual_direction} as expected by directional bias ({directional_bias})."
            )
        else:
            return "Loss", (
                f"Stock moved {actual_direction} against directional bias ({directional_bias})."
            )

    return "N/A", "Structure not recognized for Win/Loss calculation."


# =============================================================
# RESULTS TRACKER
# =============================================================

class EarningsResultsTracker:
    """
    Saves predictions before earnings and resolves
    Win/Loss 2 days after earnings using actual price data.
    Stores everything to Google Drive CSVs.
    """

    def __init__(self):
        ensure_folder()

    # ── Save prediction before earnings ───────────────────────
    def save_prediction(self, summary_df: pd.DataFrame):
        """
        Called immediately after earnings_edge_engine runs.
        Saves the prediction so it can be resolved later.
        """

        cols = [
            "Ticker", "Next Earnings", "Current Price",
            "Implied Move %", "Hist Median %", "Hist Mean %",
            "Ratio (Median)", "Ratio (Mean)", "IV Rank %",
            "Bias (Median)", "Bias (Mean)",
            "Structure (Median)", "Structure (Mean)",
            "Recommendation", "Directional Bias",
        ]

        # only keep columns that exist
        save_cols = [c for c in cols if c in summary_df.columns]
        new_rows  = summary_df[save_cols].copy()

        new_rows["prediction_date"] = date.today().isoformat()
        new_rows["resolved"]        = False
        new_rows["actual_move_pct"] = None
        new_rows["actual_direction"]= None
        new_rows["result_median"]   = None
        new_rows["result_mean"]     = None
        new_rows["result_notes"]    = None
        new_rows["resolve_date"]    = None

        if os.path.exists(PREDICTIONS_CSV):
            existing = pd.read_csv(PREDICTIONS_CSV)

            # avoid duplicate predictions for same ticker + earnings date
            combined = pd.concat([existing, new_rows], ignore_index=True)
            combined = combined.drop_duplicates(
                subset=["Ticker", "Next Earnings"], keep="last"
            )
        else:
            combined = new_rows

        combined.to_csv(PREDICTIONS_CSV, index=False)
        print(f"  Prediction saved → {PREDICTIONS_CSV}")

    # ── Resolve pending predictions ───────────────────────────
    def resolve_pending(self):
        """
        Check all unresolved predictions. For any where earnings
        have passed and 2 trading days have elapsed, fetch the
        actual price move and determine Win/Loss.
        """

        if not os.path.exists(PREDICTIONS_CSV):
            print("No predictions file found. Run the engine first.")
            return

        df      = pd.read_csv(PREDICTIONS_CSV)
        today   = pd.Timestamp.today().normalize().date()
        updated = 0

        for idx, row in df.iterrows():

            # skip already resolved
            if row["resolved"] == True:
                continue

            earnings_date = pd.to_datetime(row["Next Earnings"]).date()

            # earnings must have passed
            if earnings_date >= today:
                continue

            # need at least 2 trading days after earnings
            days_since = (today - earnings_date).days
            if days_since < 2:
                continue

            ticker = row["Ticker"]

            try:
                print(f"  Resolving {ticker} (earnings: {earnings_date})...")

                stock      = yf.Ticker(ticker)
                price_data = stock.history(period="1mo", auto_adjust=True)
                price_data.index = price_data.index.date

                trading_days = sorted(price_data.index)

                # find the event day (first trading day on or after earnings)
                event_day = next(
                    (d for d in trading_days if d >= earnings_date), None
                )
                if not event_day:
                    continue

                # find the day before earnings
                prior_days = [d for d in trading_days if d < event_day]
                if not prior_days:
                    continue
                prior_day = prior_days[-1]

                # find 2 trading days after event day
                post_days = [d for d in trading_days if d > event_day]
                if len(post_days) < 2:
                    print(f"    {ticker}: Not enough post-earnings days yet — will retry.")
                    continue

                two_day_after = post_days[1]   # 2 trading days after earnings

                prior_close     = price_data.loc[prior_day]["Close"]
                post_close      = price_data.loc[two_day_after]["Close"]

                actual_move_pct = abs(
                    (post_close - prior_close) / prior_close * 100
                )
                actual_direction = "Up" if post_close > prior_close else "Down"

                implied_move    = float(row["Implied Move %"])
                dir_bias        = str(row["Directional Bias"])

                # resolve for median structure
                result_median, notes_median = determine_win_loss(
                    str(row["Structure (Median)"]),
                    actual_move_pct,
                    implied_move,
                    dir_bias,
                    actual_direction
                )

                # resolve for mean structure
                result_mean, notes_mean = determine_win_loss(
                    str(row["Structure (Mean)"]),
                    actual_move_pct,
                    implied_move,
                    dir_bias,
                    actual_direction
                )

                # combined notes
                notes = (
                    f"[Median: {notes_median}] "
                    f"[Mean: {notes_mean}]"
                )

                # update the row
                df.at[idx, "resolved"]         = True
                df.at[idx, "actual_move_pct"]  = round(actual_move_pct, 2)
                df.at[idx, "actual_direction"]  = actual_direction
                df.at[idx, "result_median"]     = result_median
                df.at[idx, "result_mean"]       = result_mean
                df.at[idx, "result_notes"]      = notes
                df.at[idx, "resolve_date"]      = today.isoformat()

                print(f"    ✅ {ticker} resolved — "
                      f"Actual: {actual_move_pct:.1f}% {actual_direction} | "
                      f"Median: {result_median} | Mean: {result_mean}")

                updated += 1

            except Exception as e:
                print(f"    Error resolving {ticker}: {e}")
                continue

        df.to_csv(PREDICTIONS_CSV, index=False)

        if updated > 0:
            print(f"\n{updated} prediction(s) resolved and saved.")
            self._save_results(df)
        else:
            print("\nNo new predictions to resolve today.")

        return df

    # ── Save resolved results to results CSV ──────────────────
    def _save_results(self, df: pd.DataFrame):

        resolved = df[df["resolved"] == True].copy()

        if resolved.empty:
            return

        resolved.to_csv(RESULTS_CSV, index=False)
        print(f"  Results saved → {RESULTS_CSV}")

        self._update_performance(resolved)

    # ── Update performance summary ─────────────────────────────
    def _update_performance(self, resolved: pd.DataFrame):

        if resolved.empty:
            return

        records = []

        for ticker, grp in resolved.groupby("Ticker"):

            total      = len(grp)

            # median performance
            wins_m     = (grp["result_median"] == "Win").sum()
            partials_m = (grp["result_median"] == "Partial").sum()
            losses_m   = (grp["result_median"] == "Loss").sum()
            winrate_m  = round((wins_m + partials_m * 0.5) / total * 100, 1)

            # mean performance
            wins_mn    = (grp["result_mean"] == "Win").sum()
            partials_mn= (grp["result_mean"] == "Partial").sum()
            losses_mn  = (grp["result_mean"] == "Loss").sum()
            winrate_mn = round((wins_mn + partials_mn * 0.5) / total * 100, 1)

            # most common structure recommended
            top_structure_m  = grp["Structure (Median)"].mode()[0] \
                               if not grp["Structure (Median)"].empty else "N/A"
            top_structure_mn = grp["Structure (Mean)"].mode()[0] \
                               if not grp["Structure (Mean)"].empty else "N/A"

            # average actual vs implied
            avg_actual  = round(grp["actual_move_pct"].astype(float).mean(), 2)
            avg_implied = round(grp["Implied Move %"].astype(float).mean(), 2)
            avg_ratio_m = round(grp["Ratio (Median)"].astype(float).mean(), 2)

            records.append({
                "Ticker":               ticker,
                "Total Resolved":       total,
                "Wins (Median)":        wins_m,
                "Partials (Median)":    partials_m,
                "Losses (Median)":      losses_m,
                "Win Rate % (Median)":  winrate_m,
                "Wins (Mean)":          wins_mn,
                "Partials (Mean)":      partials_mn,
                "Losses (Mean)":        losses_mn,
                "Win Rate % (Mean)":    winrate_mn,
                "Top Structure (Med)":  top_structure_m,
                "Top Structure (Mean)": top_structure_mn,
                "Avg Actual Move %":    avg_actual,
                "Avg Implied Move %":   avg_implied,
                "Avg Ratio (Median)":   avg_ratio_m,
                "Last Updated":         date.today().isoformat(),
            })

        perf_df = pd.DataFrame(records).sort_values(
            "Win Rate % (Median)", ascending=False
        )

        perf_df.to_csv(PERFORMANCE_CSV, index=False)
        print(f"  Performance summary saved → {PERFORMANCE_CSV}")

    # ── Print performance report ───────────────────────────────
    def performance_report(self):

        if not os.path.exists(PERFORMANCE_CSV):
            print("No performance data yet. Run resolve_pending() first.")
            return

        df = pd.read_csv(PERFORMANCE_CSV)

        print(f"\n{'='*70}")
        print(f"  EARNINGS LAB — PERFORMANCE REPORT")
        print(f"{'='*70}")
        print(
            f"{'Ticker':<8}"
            f"{'Resolved':>10}"
            f"{'W(Med)':>8}{'P(Med)':>8}{'L(Med)':>8}"
            f"{'WR%(Med)':>10}"
            f"{'W(Mean)':>9}{'WR%(Mean)':>11}"
            f"{'AvgActual':>11}"
            f"{'AvgImplied':>11}"
        )
        print("─" * 100)

        for _, row in df.iterrows():
            print(
                f"{str(row['Ticker']):<8}"
                f"{int(row['Total Resolved']):>10}"
                f"{int(row['Wins (Median)']):>8}"
                f"{int(row['Partials (Median)']):>8}"
                f"{int(row['Losses (Median)']):>8}"
                f"{row['Win Rate % (Median)']:>10.1f}"
                f"{int(row['Wins (Mean)']):>9}"
                f"{row['Win Rate % (Mean)']:>11.1f}"
                f"{row['Avg Actual Move %']:>11.2f}"
                f"{row['Avg Implied Move %']:>11.2f}"
            )

        # overall summary
        total_resolved = df["Total Resolved"].sum()
        total_wins_m   = df["Wins (Median)"].sum()
        total_losses_m = df["Losses (Median)"].sum()
        overall_wr     = round(
            (df["Win Rate % (Median)"] * df["Total Resolved"]).sum()
            / total_resolved, 1
        ) if total_resolved > 0 else 0

        print("─" * 100)
        print(f"\n  Total Predictions Resolved : {total_resolved}")
        print(f"  Overall Win Rate (Median)  : {overall_wr}%")
        print(f"  Total Wins (Median)        : {total_wins_m}")
        print(f"  Total Losses (Median)      : {total_losses_m}")

        # best performing structure
        if os.path.exists(RESULTS_CSV):
            results = pd.read_csv(RESULTS_CSV)
            struct_perf = results.groupby("Structure (Median)").apply(
                lambda g: round(
                    ((g["result_median"] == "Win").sum()
                    + (g["result_median"] == "Partial").sum() * 0.5)
                    / len(g) * 100, 1
                )
            ).sort_values(ascending=False)

            print(f"\n  ── Win Rate by Structure (Median) ──")
            for struct, wr in struct_perf.items():
                count = (results["Structure (Median)"] == struct).sum()
                print(f"  {struct:<35} {wr:>6.1f}%  ({count} trades)")

    # ── Print full prediction history ──────────────────────────
    def prediction_history(self, ticker=None):

        if not os.path.exists(PREDICTIONS_CSV):
            print("No predictions saved yet.")
            return

        df = pd.read_csv(PREDICTIONS_CSV)

        if ticker:
            df = df[df["Ticker"] == ticker]

        if df.empty:
            print(f"No history found{' for ' + ticker if ticker else ''}.")
            return

        display_cols = [
            "Ticker", "Next Earnings", "prediction_date",
            "Implied Move %", "Hist Median %", "Ratio (Median)",
            "Structure (Median)", "Structure (Mean)", "Recommendation",
            "resolved", "actual_move_pct", "actual_direction",
            "result_median", "result_mean", "result_notes"
        ]

        cols = [c for c in display_cols if c in df.columns]

        label = f" — {ticker}" if ticker else ""
        print(f"\n{'='*70}")
        print(f"  PREDICTION HISTORY{label}")
        print(f"{'='*70}\n")
        print(df[cols].to_string(index=False))


# =============================================================
# HELPERS
# =============================================================

def find_next_trading_day(price_data, target_date, max_days=10):
    trading_days = sorted(price_data.index)
    for d in trading_days:
        if d >= target_date:
            return d
    return None


def get_bias_and_structure(ratio, iv_rank):

    if ratio > 1.5 and (iv_rank is None or iv_rank > 60):
        return "STRONG SELL VOLATILITY", "Short Iron Condor"
    elif ratio > 1.3:
        return "SELL VOLATILITY", "Short Iron Condor"
    elif ratio < 0.6 and (iv_rank is None or iv_rank < 40):
        return "STRONG BUY VOLATILITY", "Long Straddle / Strangle"
    elif ratio < 0.80:
        return "BUY VOLATILITY", "Long Straddle"
    else:
        return "NEUTRAL / FAIR", "Directional or Calendar Spread"


def consistency_label(structure_median, structure_mean):
    return "Same" if structure_median == structure_mean else "Different"


# =============================================================
# SINGLE TICKER ENGINE
# =============================================================

def earnings_edge_engine(ticker_symbol, lookback=24, save=True):
    """
    Analyze a single ticker's earnings edge and save prediction.

    Parameters
    ----------
    ticker_symbol : str
    lookback      : int   — number of past earnings reports to use
    save          : bool  — save prediction to Google Drive CSV
    """

    print(f"\nAnalyzing {ticker_symbol}...")

    ticker   = yf.Ticker(ticker_symbol)
    tracker  = EarningsResultsTracker()

    # ── 1. Earnings Dates
    earnings = ticker.get_earnings_dates(limit=lookback)

    if earnings is None or earnings.empty:
        print(f"No earnings data for {ticker_symbol}")
        return None

    earnings = earnings.reset_index()
    earnings["Earnings Date"] = pd.to_datetime(
        earnings["Earnings Date"]
    ).dt.tz_localize(None)

    today           = pd.Timestamp.today().normalize()
    past_earnings   = earnings[earnings["Earnings Date"] < today]
    future_earnings = earnings[earnings["Earnings Date"] >= today]

    if future_earnings.empty:
        print(f"No upcoming earnings for {ticker_symbol}")
        return None

    next_earnings_date = future_earnings.iloc[0]["Earnings Date"].date()

    # ── 2. Price Data & Historical Moves
    price_data = ticker.history(period="7y", auto_adjust=True)
    price_data.index = price_data.index.date

    results = []

    for _, row in past_earnings.iterrows():

        announce_date = row["Earnings Date"].date()
        event_day     = find_next_trading_day(price_data, announce_date)

        if not event_day:
            continue

        prior_day = find_next_trading_day(
            price_data, announce_date - timedelta(days=7)
        )

        if not prior_day or prior_day >= event_day:
            continue

        try:
            prior_close = price_data.loc[prior_day]["Close"]
            next_day    = find_next_trading_day(
                price_data, event_day + timedelta(days=1)
            )

            if not next_day:
                continue

            next_close   = price_data.loc[next_day]["Close"]
            two_day_pct  = (next_close - prior_close) / prior_close * 100

            results.append({
                "Earnings Date":        announce_date,
                "Event Day":            event_day,
                "Post Earnings Move %": round(abs(two_day_pct), 2),
                "Signed Move %":        round(two_day_pct, 2),
                "Direction":            "Up" if two_day_pct > 0 else "Down",
            })

        except Exception:
            continue

    if not results:
        print(f"No valid historical moves for {ticker_symbol}")
        return None

    df_moves = pd.DataFrame(results)
    n_moves  = len(df_moves)

    hist_median   = df_moves["Post Earnings Move %"].median()
    hist_mean     = df_moves["Post Earnings Move %"].mean()
    hist_std      = df_moves["Post Earnings Move %"].std()
    hist_min      = df_moves["Post Earnings Move %"].min()
    hist_max      = df_moves["Post Earnings Move %"].max()
    signed_mean   = df_moves["Signed Move %"].mean()
    signed_median = df_moves["Signed Move %"].median()

    up_moves   = df_moves[df_moves["Direction"] == "Up"].shape[0]
    down_moves = df_moves[df_moves["Direction"] == "Down"].shape[0]
    up_pct     = round(up_moves / n_moves * 100, 1)

    directional_bias = "Bullish" if up_moves > down_moves else "Bearish"

    top3 = df_moves.nlargest(3, "Post Earnings Move %")[
        ["Earnings Date", "Post Earnings Move %", "Direction"]
    ]

    # ── 3. Options + IV Rank
    expirations = ticker.options

    if not expirations:
        print(f"No options for {ticker_symbol}")
        return None

    exp_dates  = [pd.to_datetime(e).date() for e in expirations]
    valid_exps = [e for e in exp_dates if e > next_earnings_date]

    if not valid_exps:
        print(f"No valid expiration after earnings for {ticker_symbol}")
        return None

    target_exp   = min(valid_exps)
    option_chain = ticker.option_chain(str(target_exp))

    calls         = option_chain.calls
    puts          = option_chain.puts
    current_price = price_data.iloc[-1]["Close"]

    calls["distance"] = abs(calls["strike"] - current_price)
    puts["distance"]  = abs(puts["strike"]  - current_price)

    atm_call = calls.loc[calls["distance"].idxmin()]
    atm_put  = puts.loc[puts["distance"].idxmin()]

    call_mid       = (atm_call["bid"] + atm_call["ask"]) / 2
    put_mid        = (atm_put["bid"]  + atm_put["ask"])  / 2
    straddle_price = call_mid + put_mid

    implied_move_pct = (straddle_price / current_price) * 100

    current_iv = (
        atm_call["impliedVolatility"] + atm_put["impliedVolatility"]
    ) / 2 * 100

    try:
        hist_vol = (
            ticker.history(period="1y")["Close"]
            .pct_change().std() * np.sqrt(252) * 100
        )
        iv_rank = min(max((current_iv - 20) / (hist_vol * 1.5), 0), 100)
    except Exception:
        iv_rank = None

    # ── 4. Dual Bias Logic
    ratio_median = implied_move_pct / hist_median if hist_median > 0 else 1.0
    ratio_mean   = implied_move_pct / hist_mean   if hist_mean   > 0 else 1.0

    bias_median, structure_median = get_bias_and_structure(ratio_median, iv_rank)
    bias_mean,   structure_mean   = get_bias_and_structure(ratio_mean,   iv_rank)

    recommendation_consistency = consistency_label(structure_median, structure_mean)
    bias_score_median          = round((ratio_median - 1) * 100, 2)
    bias_score_mean            = round((ratio_mean   - 1) * 100, 2)

    # ── 5. Print Output
    print(f"\n{'='*55}")
    print(f"  {ticker_symbol} — Earnings Edge Analysis")
    print(f"{'='*55}")
    print(f"  Next Earnings        : {next_earnings_date}")
    print(f"  Reports Used         : {n_moves} past quarters")
    print(f"  Current Price        : ${round(current_price, 2)}")
    print(f"  Implied Move         : {round(implied_move_pct, 2)}%")
    print(f"  Straddle Price       : ${round(straddle_price, 2)}")
    if iv_rank is not None:
        print(f"  IV Rank              : {round(iv_rank, 1)}%")
    print()
    print(f"  ── Historical Move Stats ({n_moves} reports) ──")
    print(f"  Median Move          : {round(hist_median, 2)}%")
    print(f"  Mean Move            : {round(hist_mean, 2)}%")
    print(f"  Std Dev              : {round(hist_std, 2)}%")
    print(f"  Min / Max            : {round(hist_min,2)}% / {round(hist_max,2)}%")
    print(f"  Up / Down            : {up_moves} Up ({up_pct}%) / {down_moves} Down")
    print(f"  Directional Bias     : {directional_bias}")
    print()
    print(f"  ── Ratio Analysis ──")
    print(f"  Ratio (Imp/Median)   : {round(ratio_median,2)}x → {bias_median}")
    print(f"  Ratio (Imp/Mean)     : {round(ratio_mean,2)}x → {bias_mean}")
    print(f"  Structure (Median)   : {structure_median}")
    print(f"  Structure (Mean)     : {structure_mean}")
    print(f"  Recommendation       : {recommendation_consistency}")
    print()
    print(f"  ── Top 3 Largest Historical Moves ──")
    print(top3.to_string(index=False))

    # ── 6. Build Summary DataFrame
    summary_df = pd.DataFrame([{
        "Ticker":                  ticker_symbol,
        "Next Earnings":           next_earnings_date,
        "Reports Used":            n_moves,
        "Current Price":           round(current_price, 2),
        "Implied Move %":          round(implied_move_pct, 2),
        "Hist Median %":           round(hist_median, 2),
        "Hist Mean %":             round(hist_mean, 2),
        "Hist Std %":              round(hist_std, 2),
        "Hist Min %":              round(hist_min, 2),
        "Hist Max %":              round(hist_max, 2),
        "Up Count":                up_moves,
        "Down Count":              down_moves,
        "Up %":                    up_pct,
        "Directional Bias":        directional_bias,
        "Signed Mean %":           round(signed_mean, 2),
        "Signed Median %":         round(signed_median, 2),
        "Ratio (Median)":          round(ratio_median, 2),
        "Ratio (Mean)":            round(ratio_mean, 2),
        "IV Rank %":               round(iv_rank, 1) if iv_rank is not None else None,
        "Bias Score (Median)":     bias_score_median,
        "Bias Score (Mean)":       bias_score_mean,
        "Bias (Median)":           bias_median,
        "Bias (Mean)":             bias_mean,
        "Structure (Median)":      structure_median,
        "Structure (Mean)":        structure_mean,
        "Recommendation":          recommendation_consistency,
        "Top 3 Moves":             top3.to_dict(orient="records"),
    }])

    # ── 7. Save prediction to Drive
    if save:
        tracker.save_prediction(summary_df)

    return summary_df


# =============================================================
# BATCH ENGINE
# =============================================================

def earnings_edge_batch(tickers, lookback=24, save=True):
    # ── Auto-resolve any pending predictions first
    # Checks if 2+ trading days have passed since any saved earnings date
    # If yes — fetches actual price and resolves Win/Loss automatically

    tracker = EarningsResultsTracker()
    print("\nChecking for pending resolutions...")
    tracker.resolve_pending()

    if isinstance(tickers, str):
        tickers = [tickers]

    all_results = []

    for t in tickers:
        try:
            result = earnings_edge_engine(t, lookback=lookback, save=save)
            if result is not None:
                all_results.append(result)
        except Exception as e:
            print(f"Error on {t}: {e}")

    if not all_results:
        print("No results returned.")
        return pd.DataFrame()

    df = pd.concat(all_results, ignore_index=True)

    display_cols = [
        "Ticker", "Next Earnings", "Reports Used",
        "Implied Move %", "Hist Median %", "Hist Mean %",
        "Ratio (Median)", "Ratio (Mean)", "IV Rank %",
        "Bias (Median)", "Bias (Mean)",
        "Structure (Median)", "Structure (Mean)",
        "Recommendation", "Directional Bias", "Up %",
    ]

    print(f"\n{'='*55}")
    print(f"  FINAL SUMMARY TABLE — {len(df)} tickers")
    print(f"{'='*55}\n")
    print(
        df[display_cols]
        .sort_values("Ratio (Median)", ascending=False)
        .to_string(index=False)
    )

    conflicts = df[df["Recommendation"] == "Different"]
    if not conflicts.empty:
        print(f"\n  ⚠️  CONFLICTING SIGNALS — {len(conflicts)} ticker(s)")
        print(
            conflicts[[
                "Ticker", "Structure (Median)", "Structure (Mean)",
                "IV Rank %", "Directional Bias"
            ]].to_string(index=False)
        )

    return df

In [26]:
# ── CELL 1: Mount Drive once per session
mount_drive()



Mounted at /content/drive
Google Drive mounted successfully.


In [35]:
# ── CELL 2: Run analysis and save predictions
tickers = ["NVDA", "AAPL", "META", "MSFT", "AMZN"]
df = earnings_edge_batch(tickers, lookback=32)



Checking for pending resolutions...

No new predictions to resolve today.

Analyzing NVDA...

  NVDA — Earnings Edge Analysis
  Next Earnings        : 2026-05-20
  Reports Used         : 27 past quarters
  Current Price        : $225.32
  Implied Move         : 7.53%
  Straddle Price       : $16.98
  IV Rank              : 1.1%

  ── Historical Move Stats (27 reports) ──
  Median Move          : 6.28%
  Mean Move            : 7.01%
  Std Dev              : 5.78%
  Min / Max            : 0.17% / 25.85%
  Up / Down            : 18 Up (66.7%) / 9 Down
  Directional Bias     : Bullish

  ── Ratio Analysis ──
  Ratio (Imp/Median)   : 1.2x → NEUTRAL / FAIR
  Ratio (Imp/Mean)     : 1.07x → NEUTRAL / FAIR
  Structure (Median)   : Directional or Calendar Spread
  Structure (Mean)     : Directional or Calendar Spread
  Recommendation       : Same

  ── Top 3 Largest Historical Moves ──
Earnings Date  Post Earnings Move % Direction
   2023-05-24                 25.85        Up
   2020-02-13     

In [36]:
# ── CELL 3: After earnings pass — resolve Win/Loss
tracker = EarningsResultsTracker()
#tracker.resolve_pending()

In [37]:
# ── CELL 4: View full performance report
tracker.performance_report()


No performance data yet. Run resolve_pending() first.


In [39]:
# ── CELL 5: View prediction history for one ticker
tracker.prediction_history("META")


  PREDICTION HISTORY — META

Ticker Next Earnings prediction_date  Implied Move %  Hist Median %  Ratio (Median) Structure (Median)  Structure (Mean) Recommendation  resolved  actual_move_pct  actual_direction  result_median  result_mean  result_notes
  META    2026-07-29      2026-05-17           15.34            9.1            1.69  Short Iron Condor Short Iron Condor           Same     False              NaN               NaN            NaN          NaN           NaN


In [40]:
# ── CELL 6: View all predictions
tracker.prediction_history()


  PREDICTION HISTORY

Ticker Next Earnings prediction_date  Implied Move %  Hist Median %  Ratio (Median)             Structure (Median)               Structure (Mean) Recommendation  resolved  actual_move_pct  actual_direction  result_median  result_mean  result_notes
  AMZN    2026-07-30      2026-05-17           14.05           5.38            2.61              Short Iron Condor              Short Iron Condor           Same     False              NaN               NaN            NaN          NaN           NaN
  NVDA    2026-05-20      2026-05-17            7.53           6.28            1.20 Directional or Calendar Spread Directional or Calendar Spread           Same     False              NaN               NaN            NaN          NaN           NaN
  AAPL    2026-07-30      2026-05-17           10.54           4.22            2.50              Short Iron Condor              Short Iron Condor           Same     False              NaN               NaN            NaN          NaN

## Build earnings lab class

In [16]:

# =============================================================
# HELPERS
# =============================================================

def find_next_trading_day(price_data, target_date, max_days=10):
    trading_days = sorted(price_data.index)
    for d in trading_days:
        if d >= target_date:
            return d
    return None


def get_bias_and_structure(ratio, iv_rank):
    """
    Returns (bias, suggested_structure) given a ratio and iv_rank.
    Extracted as a standalone function so it can be called for
    both the median-based and mean-based ratios independently.
    """

    if ratio > 1.5 and (iv_rank is None or iv_rank > 60):
        return "STRONG SELL VOLATILITY", "Short Iron Condor"

    elif ratio > 1.3:
        return "SELL VOLATILITY", "Short Iron Condor"

    elif ratio < 0.6 and (iv_rank is None or iv_rank < 40):
        return "STRONG BUY VOLATILITY", "Long Straddle / Strangle"

    elif ratio < 0.80:
        return "BUY VOLATILITY", "Long Straddle"

    else:
        return "NEUTRAL / FAIR", "Directional or Calendar Spread"


def consistency_label(structure_median, structure_mean):
    """
    Returns 'Same' if both approaches suggest the same structure,
    'Different' otherwise.
    """
    return "Same" if structure_median == structure_mean else "Different"


# =============================================================
# SINGLE TICKER ENGINE
# =============================================================

def earnings_edge_engine(ticker_symbol, lookback=24):
    """
    Analyze a single ticker's earnings edge.

    Parameters
    ----------
    ticker_symbol : str
    lookback      : int
        Number of past earnings reports to include.
        Default raised to 24 (~6 years of quarterly reports)
        for a more statistically robust sample.
        Set higher (e.g. 40) for large caps with long history.
    """

    print(f"\nAnalyzing {ticker_symbol}...")

    ticker = yf.Ticker(ticker_symbol)

    # =====================================================
    # 1. EARNINGS DATES
    # =====================================================
    earnings = ticker.get_earnings_dates(limit=lookback)

    if earnings is None or earnings.empty:
        print(f"No earnings data for {ticker_symbol}")
        return None

    earnings = earnings.reset_index()
    earnings["Earnings Date"] = pd.to_datetime(
        earnings["Earnings Date"]
    ).dt.tz_localize(None)

    today = pd.Timestamp.today().normalize()

    past_earnings   = earnings[earnings["Earnings Date"] < today]
    future_earnings = earnings[earnings["Earnings Date"] >= today]

    if future_earnings.empty:
        print(f"No upcoming earnings for {ticker_symbol}")
        return None

    next_earnings_date = future_earnings.iloc[0]["Earnings Date"].date()

    # how many past reports are we actually using
    reports_used = len(past_earnings)
    print(f"  Using {reports_used} past earnings reports")

    # =====================================================
    # 2. PRICE DATA & HISTORICAL MOVES
    # =====================================================

    # extend to 7y to support larger lookback windows
    price_data = ticker.history(period="7y", auto_adjust=True)
    price_data.index = price_data.index.date

    results = []

    for _, row in past_earnings.iterrows():

        announce_date = row["Earnings Date"].date()
        event_day     = find_next_trading_day(price_data, announce_date)

        if not event_day:
            continue

        prior_day = find_next_trading_day(
            price_data, announce_date - timedelta(days=7)
        )

        if not prior_day or prior_day >= event_day:
            continue

        try:
            prior_close    = price_data.loc[prior_day]["Close"]
            next_day       = find_next_trading_day(
                price_data, event_day + timedelta(days=1)
            )

            if not next_day:
                continue

            next_close     = price_data.loc[next_day]["Close"]
            two_day_pct    = (next_close - prior_close) / prior_close * 100
            raw_move       = two_day_pct  # signed — preserve direction

            results.append({
                "Earnings Date":       announce_date,
                "Event Day":           event_day,
                "Post Earnings Move %": round(abs(two_day_pct), 2),
                "Signed Move %":       round(raw_move, 2),
                "Direction":           "Up" if two_day_pct > 0 else "Down",
            })

        except Exception:
            continue

    if not results:
        print(f"No valid historical moves for {ticker_symbol}")
        return None

    df_moves = pd.DataFrame(results)

    n_moves = len(df_moves)

    # ── Core statistics
    hist_median   = df_moves["Post Earnings Move %"].median()
    hist_mean     = df_moves["Post Earnings Move %"].mean()
    hist_std      = df_moves["Post Earnings Move %"].std()
    hist_min      = df_moves["Post Earnings Move %"].min()
    hist_max      = df_moves["Post Earnings Move %"].max()

    # ── Signed stats (direction-aware)
    signed_mean   = df_moves["Signed Move %"].mean()
    signed_median = df_moves["Signed Move %"].median()

    # ── Win/loss counts
    up_moves   = df_moves[df_moves["Direction"] == "Up"].shape[0]
    down_moves = df_moves[df_moves["Direction"] == "Down"].shape[0]
    up_pct     = round(up_moves / n_moves * 100, 1)

    directional_bias = "Bullish" if up_moves > down_moves else "Bearish"

    # ── Top 3 largest moves
    top3 = df_moves.nlargest(3, "Post Earnings Move %")[
        ["Earnings Date", "Post Earnings Move %", "Direction"]
    ]

    # =====================================================
    # 3. OPTIONS + IV RANK
    # =====================================================
    expirations = ticker.options

    if not expirations:
        print(f"No options for {ticker_symbol}")
        return None

    exp_dates  = [pd.to_datetime(e).date() for e in expirations]
    valid_exps = [e for e in exp_dates if e > next_earnings_date]

    if not valid_exps:
        print(f"No valid expiration after earnings for {ticker_symbol}")
        return None

    target_exp   = min(valid_exps)
    option_chain = ticker.option_chain(str(target_exp))

    calls         = option_chain.calls
    puts          = option_chain.puts
    current_price = price_data.iloc[-1]["Close"]

    calls["distance"] = abs(calls["strike"] - current_price)
    puts["distance"]  = abs(puts["strike"]  - current_price)

    atm_call = calls.loc[calls["distance"].idxmin()]
    atm_put  = puts.loc[puts["distance"].idxmin()]

    call_mid       = (atm_call["bid"] + atm_call["ask"]) / 2
    put_mid        = (atm_put["bid"]  + atm_put["ask"])  / 2
    straddle_price = call_mid + put_mid

    implied_move_pct = (straddle_price / current_price) * 100

    current_iv = (
        atm_call["impliedVolatility"] + atm_put["impliedVolatility"]
    ) / 2 * 100

    try:
        hist_vol = (
            ticker.history(period="1y")["Close"]
            .pct_change()
            .std()
            * np.sqrt(252)
            * 100
        )
        iv_rank = min(
            max((current_iv - 20) / (hist_vol * 1.5), 0), 100
        )
    except Exception:
        iv_rank = None

    # =====================================================
    # 4. DUAL BIAS LOGIC — MEDIAN AND MEAN
    # =====================================================

    ratio_median = (
        implied_move_pct / hist_median if hist_median > 0 else 1.0
    )
    ratio_mean = (
        implied_move_pct / hist_mean if hist_mean > 0 else 1.0
    )

    # bias and structure based on median
    bias_median, structure_median = get_bias_and_structure(
        ratio_median, iv_rank
    )

    # bias and structure based on mean
    bias_mean, structure_mean = get_bias_and_structure(
        ratio_mean, iv_rank
    )

    # consistency check
    recommendation_consistency = consistency_label(
        structure_median, structure_mean
    )

    # bias scores
    bias_score_median = round((ratio_median - 1) * 100, 2)
    bias_score_mean   = round((ratio_mean   - 1) * 100, 2)

    # =====================================================
    # 5. PRINT OUTPUT
    # =====================================================

    print(f"\n{'='*55}")
    print(f"  {ticker_symbol} — Earnings Edge Analysis")
    print(f"{'='*55}")
    print(f"  Next Earnings Date   : {next_earnings_date}")
    print(f"  Reports Used         : {n_moves} past quarters")
    print(f"  Current Price        : ${round(current_price, 2)}")
    print(f"  Implied Move         : {round(implied_move_pct, 2)}%")
    print(f"  Straddle Price       : ${round(straddle_price, 2)}")
    if iv_rank is not None:
        print(f"  IV Rank              : {round(iv_rank, 1)}%")
    print()
    print(f"  ── Historical Move Stats ({n_moves} reports) ──")
    print(f"  Median Move          : {round(hist_median, 2)}%")
    print(f"  Mean Move            : {round(hist_mean, 2)}%")
    print(f"  Std Dev              : {round(hist_std, 2)}%")
    print(f"  Min / Max            : {round(hist_min, 2)}% / {round(hist_max, 2)}%")
    print(f"  Up / Down            : {up_moves} Up ({up_pct}%) / {down_moves} Down")
    print(f"  Directional Bias     : {directional_bias}")
    print(f"  Signed Mean Move     : {round(signed_mean, 2)}%")
    print(f"  Signed Median Move   : {round(signed_median, 2)}%")
    print()
    print(f"  ── Ratio Analysis ──")
    print(f"  Ratio (Imp/Median)   : {round(ratio_median, 2)}x  → {bias_median}")
    print(f"  Ratio (Imp/Mean)     : {round(ratio_mean, 2)}x  → {bias_mean}")
    print(f"  Structure (Median)   : {structure_median}")
    print(f"  Structure (Mean)     : {structure_mean}")
    print(f"  Recommendation       : {recommendation_consistency}")
    print()
    print(f"  ── Top 3 Largest Historical Moves ──")
    print(top3.to_string(index=False))
    print()

    # =====================================================
    # 6. SUMMARY DATAFRAME
    # =====================================================

    summary_df = pd.DataFrame([{
        "Ticker":                  ticker_symbol,
        "Next Earnings":           next_earnings_date,
        "Reports Used":            n_moves,
        "Current Price":           round(current_price, 2),
        "Implied Move %":          round(implied_move_pct, 2),
        "Hist Median %":           round(hist_median, 2),
        "Hist Mean %":             round(hist_mean, 2),
        "Hist Std %":              round(hist_std, 2),
        "Hist Min %":              round(hist_min, 2),
        "Hist Max %":              round(hist_max, 2),
        "Up Count":                up_moves,
        "Down Count":              down_moves,
        "Up %":                    up_pct,
        "Directional Bias":        directional_bias,
        "Signed Mean %":           round(signed_mean, 2),
        "Signed Median %":         round(signed_median, 2),
        "Ratio (Median)":          round(ratio_median, 2),
        "Ratio (Mean)":            round(ratio_mean, 2),
        "IV Rank %":               round(iv_rank, 1) if iv_rank is not None else None,
        "Bias Score (Median)":     bias_score_median,
        "Bias Score (Mean)":       bias_score_mean,
        "Bias (Median)":           bias_median,
        "Bias (Mean)":             bias_mean,
        "Structure (Median)":      structure_median,
        "Structure (Mean)":        structure_mean,
        "Recommendation":          recommendation_consistency,
        "Top 3 Moves":             top3.to_dict(orient="records"),
    }])

    return summary_df


# =============================================================
# BATCH ENGINE
# =============================================================

def earnings_edge_batch(tickers, lookback=24):
    """
    Run earnings edge analysis on a list of tickers.

    Parameters
    ----------
    tickers  : list or str
    lookback : int
        Number of past earnings to fetch per ticker.
        Default 24 = ~6 years of quarterly data.
        Increase to 40 for the deepest available history
        on large caps (AAPL, MSFT etc go back further).
    """

    if isinstance(tickers, str):
        tickers = [tickers]

    all_results = []

    for t in tickers:
        try:
            result = earnings_edge_engine(t, lookback=lookback)
            if result is not None:
                all_results.append(result)
        except Exception as e:
            print(f"Error on {t}: {e}")

    if not all_results:
        print("No results returned.")
        return pd.DataFrame()

    df = pd.concat(all_results, ignore_index=True)

    # ── Summary table
    display_cols = [
        "Ticker",
        "Next Earnings",
        "Reports Used",
        "Implied Move %",
        "Hist Median %",
        "Hist Mean %",
        "Ratio (Median)",
        "Ratio (Mean)",
        "IV Rank %",
        "Bias (Median)",
        "Bias (Mean)",
        "Structure (Median)",
        "Structure (Mean)",
        "Recommendation",       # Same / Different
        "Directional Bias",
        "Up %",
    ]

    print(f"\n{'='*55}")
    print(f"  FINAL SUMMARY TABLE — {len(df)} tickers")
    print(f"{'='*55}\n")

    print(
        df[display_cols]
        .sort_values("Ratio (Median)", ascending=False)
        .to_string(index=False)
    )

    # ── Flag conflicts — where median and mean disagree
    conflicts = df[df["Recommendation"] == "Different"]

    if not conflicts.empty:
        print(f"\n{'='*55}")
        print(f"  ⚠️  CONFLICTING SIGNALS — {len(conflicts)} ticker(s)")
        print(f"  Mean and Median suggest different structures.")
        print(f"  Use IV Rank and Directional Bias to decide.")
        print(f"{'='*55}")
        print(
            conflicts[[
                "Ticker", "Structure (Median)", "Structure (Mean)",
                "IV Rank %", "Directional Bias"
            ]].to_string(index=False)
        )

    return df



In [20]:
# =============================================================
# EXAMPLE USAGE
# =============================================================

if __name__ == "__main__":

    tickers = ["NVDA", "HD", "KEYS", "VSAT", "ADI","LOW", "INTU", "TGT", "WMT", "ROST","TTWO", "CPRT", "WDAY", "MNSO", "IMPP" ]

    # ── Single ticker deep dive
    #result = earnings_edge_engine("NVDA", lookback=32)

    # ── Batch with extended lookback
    df = earnings_edge_batch(tickers, lookback=32)

df


Analyzing NVDA...
  Using 49 past earnings reports

  NVDA — Earnings Edge Analysis
  Next Earnings Date   : 2026-05-20
  Reports Used         : 27 past quarters
  Current Price        : $225.32
  Implied Move         : 7.53%
  Straddle Price       : $16.98
  IV Rank              : 1.1%

  ── Historical Move Stats (27 reports) ──
  Median Move          : 6.28%
  Mean Move            : 7.01%
  Std Dev              : 5.78%
  Min / Max            : 0.17% / 25.85%
  Up / Down            : 18 Up (66.7%) / 9 Down
  Directional Bias     : Bullish
  Signed Mean Move     : 2.89%
  Signed Median Move   : 2.72%

  ── Ratio Analysis ──
  Ratio (Imp/Median)   : 1.2x  → NEUTRAL / FAIR
  Ratio (Imp/Mean)     : 1.07x  → NEUTRAL / FAIR
  Structure (Median)   : Directional or Calendar Spread
  Structure (Mean)     : Directional or Calendar Spread
  Recommendation       : Same

  ── Top 3 Largest Historical Moves ──
Earnings Date  Post Earnings Move % Direction
   2023-05-24                 25.85       

,Ticker,Next Earnings,Reports Used,Current Price,Implied Move %,Hist Median %,Hist Mean %,Hist Std %,Hist Min %,Hist Max %,...,Ratio (Mean),IV Rank %,Bias Score (Median),Bias Score (Mean),Bias (Median),Bias (Mean),Structure (Median),Structure (Mean),Recommendation,Top 3 Moves
0,NVDA,2026-05-20,27,225.32,7.53,6.28,7.01,5.78,0.17,25.85,...,1.07,1.1,19.96,7.40,NEUTRAL / FAIR,NEUTRAL / FAIR,Directional or Calendar Spread,Directional or Calendar Spread,Same,"[{'Earnings Date': 2023-05-24, 'Post Earnings ..."
1,HD,2026-05-19,28,297.51,4.98,2.95,4.00,3.02,0.49,12.68,...,1.24,0.9,69.20,24.46,SELL VOLATILITY,NEUTRAL / FAIR,Short Iron Condor,Directional or Calendar Spread,Different,"[{'Earnings Date': 2022-02-22, 'Post Earnings ..."
2,KEYS,2026-05-19,28,349.01,15.00,5.28,7.74,7.26,0.08,28.62,...,1.94,0.7,184.08,93.75,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2026-02-23, 'Post Earnings ..."
3,VSAT,2026-05-19,28,69.50,22.16,9.72,13.00,16.62,0.06,89.38,...,1.70,0.6,127.97,70.44,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-08-05, 'Post Earnings ..."
4,ADI,2026-05-20,28,417.49,7.05,3.19,5.01,3.99,0.06,18.86,...,1.41,1.1,121.48,40.91,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-02-19, 'Post Earnings ..."
5,LOW,2026-05-20,28,218.42,5.36,3.94,5.16,4.27,0.05,15.57,...,1.04,0.8,36.13,3.76,SELL VOLATILITY,NEUTRAL / FAIR,Short Iron Condor,Directional or Calendar Spread,Different,"[{'Earnings Date': 2019-08-21, 'Post Earnings ..."
6,INTU,2026-05-20,28,393.00,9.13,3.96,4.72,3.31,0.52,14.08,...,1.94,1.2,130.39,93.61,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2021-11-18, 'Post Earnings ..."
7,TGT,2026-05-20,28,121.54,8.37,8.10,9.80,8.20,0.14,30.83,...,0.85,1.3,3.35,-14.59,NEUTRAL / FAIR,NEUTRAL / FAIR,Directional or Calendar Spread,Directional or Calendar Spread,Same,"[{'Earnings Date': 2019-08-21, 'Post Earnings ..."
8,WMT,2026-05-21,27,131.45,5.03,2.70,4.18,3.84,0.65,17.93,...,1.20,0.9,86.24,20.37,SELL VOLATILITY,NEUTRAL / FAIR,Short Iron Condor,Directional or Calendar Spread,Different,"[{'Earnings Date': 2022-05-17, 'Post Earnings ..."
9,ROST,2026-05-21,28,212.75,6.84,3.72,4.90,4.11,0.09,19.36,...,1.40,1.3,84.09,39.54,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2022-05-19, 'Post Earnings ..."


In [18]:
# Sort by richest volatility (best for selling premium)
df.sort_values(by="Ratio (Mean)", ascending=False)

,Ticker,Next Earnings,Reports Used,Current Price,Implied Move %,Hist Median %,Hist Mean %,Hist Std %,Hist Min %,Hist Max %,...,Ratio (Mean),IV Rank %,Bias Score (Median),Bias Score (Mean),Bias (Median),Bias (Mean),Structure (Median),Structure (Mean),Recommendation,Top 3 Moves
3,MSFT,2026-07-29,28,421.92,13.28,2.83,3.59,2.93,0.11,13.62,...,3.70,0.4,370.25,270.01,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-04-30, 'Post Earnings ..."
1,AAPL,2026-07-30,28,300.23,10.54,4.22,4.33,2.99,0.01,14.45,...,2.44,0.2,149.81,143.58,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2020-07-30, 'Post Earnings ..."
4,AMZN,2026-07-30,28,264.14,14.05,5.38,5.87,4.23,0.30,16.19,...,2.39,0.3,161.31,139.47,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2022-04-28, 'Post Earnings ..."
2,META,2026-07-29,28,614.23,15.34,9.10,9.92,7.92,0.54,33.41,...,1.55,0.3,68.71,54.70,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2023-02-01, 'Post Earnings ..."
0,NVDA,2026-05-20,27,225.32,7.53,6.28,7.01,5.78,0.17,25.85,...,1.07,1.1,19.96,7.40,NEUTRAL / FAIR,NEUTRAL / FAIR,Directional or Calendar Spread,Directional or Calendar Spread,Same,"[{'Earnings Date': 2023-05-24, 'Post Earnings ..."


In [19]:
# Or see only the ones worth trading
df[df["Ratio (Mean)"] > 1.25]   # Good for selling vol

,Ticker,Next Earnings,Reports Used,Current Price,Implied Move %,Hist Median %,Hist Mean %,Hist Std %,Hist Min %,Hist Max %,...,Ratio (Mean),IV Rank %,Bias Score (Median),Bias Score (Mean),Bias (Median),Bias (Mean),Structure (Median),Structure (Mean),Recommendation,Top 3 Moves
1,AAPL,2026-07-30,28,300.23,10.54,4.22,4.33,2.99,0.01,14.45,...,2.44,0.2,149.81,143.58,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2020-07-30, 'Post Earnings ..."
2,META,2026-07-29,28,614.23,15.34,9.10,9.92,7.92,0.54,33.41,...,1.55,0.3,68.71,54.70,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2023-02-01, 'Post Earnings ..."
3,MSFT,2026-07-29,28,421.92,13.28,2.83,3.59,2.93,0.11,13.62,...,3.70,0.4,370.25,270.01,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2025-04-30, 'Post Earnings ..."
4,AMZN,2026-07-30,28,264.14,14.05,5.38,5.87,4.23,0.30,16.19,...,2.39,0.3,161.31,139.47,SELL VOLATILITY,SELL VOLATILITY,Short Iron Condor,Short Iron Condor,Same,"[{'Earnings Date': 2022-04-28, 'Post Earnings ..."
